# 推理服务与量化补充线 · 第 4/8 课：Speculative Decoding、Prefix Reuse 与正确性

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现候选 token 的最长接受前缀，并说明精确 speculative decoding 的校正条件。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

prefix cache 复用已确定前缀；speculative decoding 用便宜 draft 一次提出多个未来 token，再由 target 并行验证，二者解决不同重复计算。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Speculative decoding 是保持 target 模型输出分布不变的加速协议，而不是用小模型替代大模型；draft 负责提案，target 保留最终决定权。

### 数据与控制如何流动

target 对 draft tokens 计算概率；接受规则依算法而定。教学代码只计算贪心模式下与 target argmax 连续一致的最长前缀，第一次不一致立即停止。

### 正确性条件与常见误区

一般采样场景不能简单比较 argmax 并丢弃不一致 token；要按概率比接受并从校正分布采样，才能保持 target 分布。随机数和 EOS 处理也必须一致。

### 性能、成本与工程取舍

draft 越快、接受率越高、一次提议越长，越可能加速；低接受率会浪费 target 验证和 draft 成本，长 proposal 还增加尾延迟。

## 具体演示

draft=[4,8,9,2]、target greedy=[4,8,3,1] 时接受前 2 个；第三个不一致后，第四个即使相同也不能跳过中间状态单独接受。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐贪心 speculative decoding 的连续接受长度。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def accepted_prefix_length(draft_tokens, target_tokens):
    if len(draft_tokens) != len(target_tokens):
        raise ValueError("verification lengths must match")
    accepted = 0
    for draft, target in zip(draft_tokens, target_tokens):
        if draft != target:
            break
        # TODO：只累计从开头连续匹配的 token。
        ______
    return accepted

assert accepted_prefix_length([4, 8, 9, 2], [4, 8, 3, 2]) == 2
assert accepted_prefix_length([], []) == 0


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么“不一致后继续接受后面碰巧相同的 token”是错误的？

**你的答案：**


### Q2

一般温度采样中只比较 draft/target argmax，为什么会改变输出分布？

**你的答案：**


### Q3

接受率很高却没有加速，你会检查哪些成本？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def accepted_prefix_length(draft_tokens, target_tokens):
    if len(draft_tokens) != len(target_tokens):
        raise ValueError("verification lengths must match")
    accepted = 0
    for draft, target in zip(draft_tokens, target_tokens):
        if draft != target:
            break
        accepted += 1
    return accepted

assert accepted_prefix_length([4, 8, 9, 2], [4, 8, 3, 2]) == 2
assert accepted_prefix_length([], []) == 0


### Q1 参考答案

自回归 token 的条件分布依赖全部前缀。第三个 token 改变后，draft 的第四个 token 是在旧前缀下提出的，不能视为 target 新前缀下的合法验证结果。接受集合必须是从头连续的前缀。

### Q2 参考答案

argmax 接受忽略两分布的概率质量，只保留模式一致事件，会系统性偏向高概率 token。精确算法需按 `min(1,p/q)` 接受，并在拒绝时从校正剩余分布采样。

### Q3 参考答案

检查 draft 自身延迟、proposal 长度、target 验证 batch/shape、KV 复制、调度和同步开销，以及是否 decode 已被其他共享资源限制。接受 token 数只是收益的一部分。

## 参考资料

- [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)

API 与平台能力会演进；部署前应按目标版本重新核对。